In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 
import numpy as np
import pandas as pd
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import random
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import shutil

sys.path.append('../')
from meta_fusion.benchmarks import *
from meta_fusion.methods import *
from meta_fusion.models import *
from meta_fusion.utils import *
from meta_fusion.third_party import *
from meta_fusion.synthetic_data import PrepareSyntheticData
from meta_fusion.config import *
from meta_fusion.methodsextra import *
from meta_fusion.methodsextra_new import *

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [ ]:
#########################
# Experiment parameters #
#########################
if True:
    # Parse input arguments
    print ('Number of arguments:', len(sys.argv), 'arguments.')
    print ('Argument List:', str(sys.argv))
    if len(sys.argv) != 2:
        print("Error: incorrect number of parameters.")
        quit()

    seed = 1234
    print(seed)

# Fixed data parameters
repetition=1

# Data model parameters
n = 2000
dim_modalities = [200, 300, 100]
# dim_modalities = [100, 150, 50]

dim_latent = [20, 30, 10, 0] # last one is the shared component 
# noise_ratios = [0.6, 0.1, 0.1, 0, 0, 0.9] # 19 March:Correlation pairwise/correlation with the output, if y >0, more missing

noise_ratios = [0.6, 0.1, 0.1]
# noise_ratios = [0.2, 0.2, 0.2]

trans_type = ["linear", "quadratic", "quadratic", "linear"] #last one is shared
# trans_type = ["quadratic", "quadratic", "quadratic", "quadratic"] # New: last one is shared, change to quadratic

mod_prop = [1, 1, 1, 0, 0]
interactive_prop = 0

missing_value = 100.0
fractions = [0.95, 0.5, 0.5] # presence percentages

# mod_outs = [[0, 200, 300, 400, 500], [0, 100, 200, 300, 400]]
# mod_outs = [[0, 500], [0, 400]]
num_modalities = len(dim_modalities)
print('num_modalities', num_modalities)
combined_hiddens = [128, 64] # only used for benchmarks
mod_hiddens = [[256], [256], [256]] # hidden layer for each modality

# data parameters
data_name = 'regression'
exp_name = data_name + "_" + "linear_early"
output_dim = 1  # specify the output dimension for regression


extractor_type = 'separate'
separate=True
is_mod_static=[False]*num_modalities
freeze_mod_extractors=[False]*num_modalities

# Load default model configurations 
config = load_config('../experiments_synthetic/config.json')
extractor_config = load_config('../experiments_synthetic/config_extractor.json')

# Model files directory
ckpt_dir = f"./checkpoints/{exp_name}/seed{seed}/"
config['ckpt_dir'] = extractor_config['ckpt_dir'] = ckpt_dir

# Update other training parameters
config['output_dim'] = extractor_config['output_dim'] = output_dim
config["init_lr"] = 0.001
config["ensemble_methods"] = [
        "simple_average",
        "weighted_average",
        # "meta_learner",
        "greedy_ensemble"
        ]
extractor_config["init_lr"] = [0.001] * num_modalities
extractor_config["weight_decay"] = [0] * num_modalities

#####################
#    Load Dataset   #
#####################
data_preparer = PrepareSyntheticData(data_name = data_name, test_size = 0.2, val_size = 0.2)
print(f"Finished generating {exp_name} dataset.")
sys.stdout.flush() 


###############
# Output file #
###############
# NOTE: set this to "imputation" (missing modalities kept, trainer handles missing)
# or "filtered" (only fully-observed samples kept)
# EXPERIMENT_VARIANT = "imputation"
EXPERIMENT_VARIANT = "filtered"


# Fractions (presence percentages) used for missingness experiments
fractions = [1, 0.8, 0.6]

outdir = f"./results/{exp_name}/"
os.makedirs(outdir, exist_ok=True)

# Derive a compact transform label (e.g., "linear" or "quadratic").
# If not all transforms are the same, encode them in order as m1_.._mK_.._shared_..
if len(set(trans_type)) == 1:
    trans_label = trans_type[0]
else:
    # trans_type is expected to be length (num_modalities + 1) where last entry is shared
    per_mod = [f"m{i+1}{t}" for i, t in enumerate(trans_type[:num_modalities])]
    shared = f"shared{trans_type[num_modalities]}" if len(trans_type) > num_modalities else "shared?"
    trans_label = "_".join(per_mod + [shared])


# Header for results file
def add_header(results):
    results['extractor']=extractor_type
    results['weight_type'] = config['divergence_weight_type'] 
    return results




Number of arguments: 2 arguments.
Argument List: ['/opt/anaconda3/envs/fusion_stable310/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/parnian/Library/Jupyter/runtime/kernel-v3e24b4b445fe2ccd64eae285e4452fab757c3b2a5.json']
1234
num_modalities 3
Finished generating regression_linear_early dataset.


In [ ]:
#----------------#
# Split dataset  #
#----------------#
missing_value = 10.0
train_loader, val_loader, test_loader, oracle_train_loader, oracle_val_loader, oracle_test_loader =\
data_preparer.get_data_loaders(n, trans_type=trans_type, mod_prop=mod_prop, 
                                interactive_prop = interactive_prop,
                                dim_modalities=dim_modalities, dim_latent=dim_latent,
                                noise_ratios=noise_ratios, random_state=1234)
fractions = [1, 0.8, 0.6] # presence percentages
train_miss, val_miss, test_miss = data_preparer.apply_missing_modalities(
    train_loader, val_loader, test_loader, modality_fractions=fractions, random_state=0,
    missing_value=missing_value
)                     ### based on presence fraction of each modality, randomly mask some values to missing_value.          
train_all_modalities, val_all_modalities, test_all_modalities = data_preparer.filter_fully_observed(
    train_miss, val_miss, test_miss, missing_value=missing_value
)  #### Only keep the datapoints that have all modalities present. 

In [4]:
*modalities, y = train_loader.dataset[0]
print(len(modalities), len(modalities[0]), len(modalities[1]), len(modalities[2]), y)


3 500 400 100 tensor([-9.2496])


In [5]:
meta_cohort = Cohorts_new(dim_modalities = dim_modalities, num_modalities= num_modalities, mod_hiddens = mod_hiddens, output_dim=output_dim)


In [13]:
cohort_models = meta_cohort.get_cohort_models()
_, dim_pairs = meta_cohort.get_cohort_info()
print(cohort_models, dim_pairs)

[MLP_Net(
  (model): Sequential(
    (0): Linear(in_features=500, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=1, bias=True)
  )
), MLP_Net(
  (model): Sequential(
    (0): Linear(in_features=400, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=1, bias=True)
  )
), MLP_Net(
  (model): Sequential(
    (0): Linear(in_features=100, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=1, bias=True)
  )
)] [500, 400, 100]


In [16]:
jointmodel = Trainer_Joint_new(config, cohort_models, [train_miss, val_miss]) # New trainer function. 
jointmodel.train('marginal', missing_value=missing_value) 
res = jointmodel.test(test_miss, missing_value=missing_value) 

Start training student cohort...

Epoch: 1/10 - LR: 0.001000


100%|██████████| 1280/1280 [00:00<00:00, 1903.75it/s, loss=18.4219, batch_time=0.034s]



Epoch: 2/10 - LR: 0.001000


100%|██████████| 1280/1280 [00:00<00:00, 2289.00it/s, loss=16.7726, batch_time=0.028s]



Epoch: 3/10 - LR: 0.001000


...

meta_learner: train task loss: 57.487 - val task loss: 91.165 [*] Best so far


1280it [00:00, 5908.19it/s]           


meta_learner: train task loss: 55.852 - val task loss: 83.567 [*] Best so far


1280it [00:00, 5836.96it/s]           


meta_learner: train task loss: 52.745 - val task loss: 82.635 [*] Best so far
Done!
Selecting greedy ensemble on the best cohort...
Pruned 1 worst models, keeping 2 models
Initial best models: [0, 1] with losses: [tensor(184.4302, grad_fn=<MseLossBackward0>), tensor(3500.3726, grad_fn=<MseLossBackward0>)]
Done!
Method: (simple_average), Test_MSE: 58.96955490112305
Method: (weighted_average), Test_MSE: 61.7009391784668
Method: (meta_learner), Test_MSE: 74.01079559326172
Method: (greedy_ensemble), Test_MSE: 92.37108612060547
Method: (best_single), Test_MSE: 190.04820251464844
Method: (cohort), Test_MSE: [190.04820251464844, 272.7131042480469, 205.59217834472656]


In [14]:
### With imputation of missing values. 

In [3]:
#####################
# Define Experiment #
#####################
def run_single_experiment(config, extractor_config, n, random_state, mod_hiddens,
                          run_oracle=False, run_coop=True, run_all_at_once=False):


    config['random_state'] = random_state
    extractor_config['random_state'] = random_state
    res_list = []
    best_rho = {}
    cohort_pairs = {}
    ens_idxs = {}
    cluster_idxs = {}


    #----------------#
    # Split dataset  #
    #----------------#
    train_loader, val_loader, test_loader, oracle_train_loader, oracle_val_loader, oracle_test_loader =\
    data_preparer.get_data_loaders(n, trans_type=trans_type, mod_prop=mod_prop, 
                                    interactive_prop = interactive_prop,
                                    dim_modalities=dim_modalities, dim_latent=dim_latent,
                                    noise_ratios=noise_ratios, random_state=random_state)
    # Get data info

    # fractions = [1.0, 1.0, 1.0] # presence percentages
    train_loader, val_loader, test_loader = data_preparer.apply_missing_modalities(
        train_loader, val_loader, test_loader, modality_fractions=fractions, random_state=0,
        missing_value=missing_value
    )                     ### based on presence fraction of each modality, randomly mask some values to missing_value.          

    n_train = len(train_loader.dataset)
    n_val = len(val_loader.dataset)
    n_test = len(test_loader.dataset)
    n = n_train + n_val + n_test

    print(f"Finished splitting {data_name} dataset. Data information are summarized below:\n"
            f"Modality dimensions: {dim_modalities}\n"
            f"Data size: {n}\n"
            f"Train size: {n_train}\n"
            f"Val size: {n_val}\n"
            f"Test size: {n_test}")
    sys.stdout.flush() 

    #------------------#
    # Benchmark models #
    #------------------#
    bm_extractor = Extractors([[d,0] for d in dim_modalities], dim_modalities, train_loader, val_loader)
    _ = bm_extractor.get_dummy_extractors()
    bm_cohort = Cohorts(extractors=bm_extractor, combined_hidden_layers=combined_hiddens, output_dim=output_dim)
    meta_cohort = Cohorts_new(dim_modalities = dim_modalities, num_modalities= num_modalities, mod_hiddens  = mod_hiddens, output_dim=output_dim)
    bm_models = bm_cohort.get_cohort_models()
    _, bm_dims = bm_cohort.get_cohort_info()
    bm = Benchmarks(config, bm_models, bm_dims, [train_loader, val_loader])
    bm.train()
    res = bm.test(test_loader)
    res_list.append(res)
    print(f"Finished running basic benchmarks!")

    #------------------------------#
    #  Train and test Meta Fuse    #
    #------------------------------#
    cohort_models = meta_cohort.get_cohort_models()
    _, dim_pairs = meta_cohort.get_cohort_info()
    ###### Only change the two following.
    metafuse = Trainer_new(config, cohort_models, [train_loader, val_loader]) # New trainer function. 
    metafuse.train() 
    res = metafuse.test(test_loader) # No need to change test: simple_averaging in test_regresion() also # performance of each student on the test data, cohort_accuracy: automaticall printed and stored. 
    res = {f"metafusion_{k}": v for k, v in res.items()}
    res_list.append(res)
    metafuse.train_ablation() # This is just late fusion with student cohort.
    res = metafuse.test_ablation(test_loader) # I don't need this, no need to have different rhos.
    res = {f"indep_{k}": v for k, v in res.items()}
    res_list.append(res)

    best_rho['metafusion'] = metafuse.best_rho
    cohort_pairs['metafusion'] = dim_pairs
    cohort_pairs['indep'] = dim_pairs

    if "greedy_ensemble" in config["ensemble_methods"]:
        ens_idxs['metafusion_greedy_ensemble'] = metafuse.ens_idxs  

    if config['divergence_weight_type'] == "clustering":
        cluster_idxs['metafusion'] = metafuse.cluster_idxs

    print(f"Finished running meta fusion!")


    #----------------------------#
    # Proposed model: Joint train#
    #----------------------------#
    joint_cohort = Cohorts_new(dim_modalities = dim_modalities, num_modalities= num_modalities, mod_hiddens  = mod_hiddens, output_dim=output_dim)

    # ------------------------------#
    #  Train and test Joint train  #
    # ------------------------------#
    cohort_models = joint_cohort.get_cohort_models()
    _, dim_pairs = joint_cohort.get_cohort_info()
    ###### Only change the two following.
    jointmodel = Trainer_Joint_new(config, cohort_models, [train_loader, val_loader]) # New trainer function. 
    jointmodel.train('marginal', missing_value=missing_value) 
    res = jointmodel.test(test_loader, missing_value=missing_value) # No need to change test: simple_averaging in test_regresion() also # performance of each student on the test data, cohort_accuracy: automaticall printed and stored. 
    res = {f"jointlearning_{k}": v for k, v in res.items()}
    res_list.append(res)
    cohort_pairs['cohort'] = dim_pairs

    if "greedy_ensemble" in config["ensemble_methods"]:
        ens_idxs['jointlearning_greedy_ensemble'] = jointmodel.ens_idxs  


    print(f"Finished running joint fusion!")


    #----------------------------#
    # Proposed model: Shapley train#
    #----------------------------#

    joint_cohort = Cohorts_new(dim_modalities = dim_modalities, num_modalities= num_modalities, mod_hiddens  = mod_hiddens, output_dim=output_dim)

    #------------------------------#
    #  Train and test shapley train  #
    #------------------------------#
    cohort_models = joint_cohort.get_cohort_models()
    _, dim_pairs = joint_cohort.get_cohort_info()
    ###### Only change the two following.
    jointmodel = Trainer_Joint_new(config, cohort_models, [train_loader, val_loader]) # New trainer function. 
    jointmodel.train('shapley', missing_value=missing_value) 
    res = jointmodel.test(test_loader,missing_value=missing_value) # No need to change test: simple_averaging in test_regresion() also # performance of each student on the test data, cohort_accuracy: automaticall printed and stored. 
    res = {f"shapley_{k}": v for k, v in res.items()}
    res_list.append(res)
    cohort_pairs['cohort'] = dim_pairs

    if "greedy_ensemble" in config["ensemble_methods"]:
        ens_idxs['shapley_greedy_ensemble'] = jointmodel.ens_idxs  


    print(f"Finished running shapley fusion!")        

    results = []
    for i, res in enumerate(res_list):
        for method, val in res.items():
            results.append({'Method': method, 'Test_metric': val, 
                            'best_rho':best_rho.get(method.split('_')[0]), 'cohort_pairs':cohort_pairs.get(method.split('_')[0]),
                            'ensemble_idxs': ens_idxs.get(method), 'cluster_idxs': cluster_idxs.get(method.split('_')[0])})

    results = pd.DataFrame(results)
    results['random_state']=random_state
    results["dim_modalities"] = [dim_modalities] * len(results)
    results['n'] = n
    results['n_train'] = n_train
    results['n_val'] = n_val
    results['n_test'] = n_test 

    return results




In [4]:
#####################
#  Run Experiments  #
#####################
results = []

for i in tqdm(range(1, repetition+1), desc="Repetitions", leave=True, position=0):
    print(f'Running with repetition {i}...')
    random_state = repetition * (seed-1) + i
    # print(random_state)
    set_random_seed(random_state)

    # Run experiment
    tmp = run_single_experiment(config, extractor_config, n, random_state, mod_hiddens,
                                run_oracle=False, run_coop=True, run_all_at_once=False)
    
    results.append(tmp)


results = pd.concat(results, ignore_index=True)

add_header(results)

Repetitions:   0%|          | 0/1 [00:00<?, ?it/s]

Running with repetition 1...
Finished splitting regression dataset. Data information are summarized below:
Modality dimensions: [200, 300, 100]
Data size: 2000
Train size: 1280
Val size: 320
Test size: 400


/Users/parnian/Desktop/Github/Multimodal-ML/notebooks/../meta_fusion/utils.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.data = torch.tensor(data, dtype=torch.float32)


Start training benchmark models...
Training with disagreement penalty = 0

Epoch: 1/10 - LR: 0.001000


100%|██████████| 1280/1280 [00:00<00:00, 2824.58it/s]


...

100%|██████████| 1280/1280 [00:00<00:00, 1447.39it/s, avg_loss=250.4232, batch_time=0.044s]



Epoch: 10/10 - LR: 0.001000
Using exact Shapley computation


Repetitions: 100%|██████████| 1/1 [00:30<00:00, 30.53s/it]

Finished training student cohort!
Selecting greedy ensemble on the best cohort...
Pruned 1 worst models, keeping 2 models
Initial best models: [1, 0] with losses: [tensor(357.0607, grad_fn=<MseLossBackward0>), tensor(463.9885, grad_fn=<MseLossBackward0>)]
Done!
Method: (simple_average), Test_MSE: 290.61163330078125
Method: (weighted_average), Test_MSE: 285.4935607910156
Method: (greedy_ensemble), Test_MSE: 307.0782775878906
Method: (best_single), Test_MSE: 341.9085693359375
Method: (cohort), Test_MSE: [428.3910217285156, 341.1065368652344, 361.194091796875]
Finished running shapley fusion!


,Method,Test_metric,best_rho,cohort_pairs,ensemble_idxs,cluster_idxs,random_state,dim_modalities,n,n_train,n_val,n_test,extractor,weight_type
0,modality_1,359.345306,NaN,None,None,None,1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
1,modality_2,332.801178,NaN,None,None,None,1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
2,modality_3,360.946198,NaN,None,None,None,1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
3,early_fusion,315.762024,NaN,None,None,None,1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
4,late_fusion,312.86676,NaN,None,None,None,1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
5,metafusion_simple_average,323.327423,0.99,"[200, 300, 100]",None,"[0, 1]",1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
6,metafusion_weighted_average,322.389191,0.99,"[200, 300, 100]",None,"[0, 1]",1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
7,metafusion_greedy_ensemble,318.656281,0.99,"[200, 300, 100]","[1, 0]","[0, 1]",1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
8,metafusion_best_single,329.667877,0.99,"[200, 300, 100]",None,"[0, 1]",1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
9,metafusion_cohort,"[354.6806640625, 329.6678771972656, 361.209106...",0.99,"[200, 300, 100]",None,"[0, 1]",1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering


: 

: 

: 

In [6]:
EXPERIMENT_VARIANT = "imputation"

# Build a filename that encodes the key experiment parameters
outfile_name = (
    f"{EXPERIMENT_VARIANT}_"
    f"dim_modalities{dim_modalities}_"
    f"noise{noise_ratios}_"
    f"trans{trans_label}_"
    f"fractions{fractions}_"
    f"seed{seed}"
)
# sanitize characters that are annoying in filenames
outfile_name = (
    outfile_name.replace(" ", "")
    .replace("[", "(")
    .replace("]", ")")
    .replace(",", "-")
)

outfile = outdir + outfile_name + ".txt"
print("Output file: {:s}".format(outfile), end="\n")
sys.stdout.flush()


#####################
#    Save Results   #
#####################
results.to_csv(outfile, index=False)
print("\nResults written to {:s}\n".format(outfile))
sys.stdout.flush()

# After the job is done, remove the model directory to free up space
if os.path.exists(ckpt_dir):
    print(f"Deleting the model checkpoint directory: {ckpt_dir}")
    shutil.rmtree(ckpt_dir)
    print(f"Model checkpoint directory {ckpt_dir} has been deleted.")


Output file: ./results/regression_linear_early/imputation_dim_modalities(200-300-100)_noise(0.6-0.1-0.1)_transm1linear_m2quadratic_m3quadratic_sharedlinear_fractions(1-0.8-0.6)_seed1234.txt

Results written to ./results/regression_linear_early/imputation_dim_modalities(200-300-100)_noise(0.6-0.1-0.1)_transm1linear_m2quadratic_m3quadratic_sharedlinear_fractions(1-0.8-0.6)_seed1234.txt



In [ ]:
### Filtering the data

In [3]:
#####################
# Define Experiment #
#####################
def run_single_experiment_filter(config, extractor_config, n, random_state, mod_hiddens,
                          run_oracle=False, run_coop=True, run_all_at_once=False):


    config['random_state'] = random_state
    extractor_config['random_state'] = random_state
    res_list = []
    best_rho = {}
    cohort_pairs = {}
    ens_idxs = {}
    cluster_idxs = {}


    #----------------#
    # Split dataset  #
    #----------------#
    train_loader, val_loader, test_loader, oracle_train_loader, oracle_val_loader, oracle_test_loader =\
    data_preparer.get_data_loaders(n, trans_type=trans_type, mod_prop=mod_prop, 
                                    interactive_prop = interactive_prop,
                                    dim_modalities=dim_modalities, dim_latent=dim_latent,
                                    noise_ratios=noise_ratios, random_state=random_state)
    # Get data info
    train_miss, val_miss, test_miss = data_preparer.apply_missing_modalities(
        train_loader, val_loader, test_loader, modality_fractions=fractions, random_state=0,
        missing_value=missing_value
    )                     ### based on presence fraction of each modality, randomly mask some values to missing_value.          
    train_loader, val_loader, test_loader = data_preparer.filter_fully_observed(
        train_miss, val_miss, test_miss, missing_value=missing_value
    )  #### Only keep the datapoints that have all modalities present. 
    # After applying missingness / filtering, sizes should come from the loaders
    n_train = len(train_loader.dataset)
    n_val = len(val_loader.dataset)
    n_test = len(test_loader.dataset)
    n = n_train + n_val + n_test

    print(f"Finished splitting {data_name} dataset. Data information are summarized below:\n"
            f"Modality dimensions: {dim_modalities}\n"
            f"Data size: {n}\n"
            f"Train size: {n_train}\n"
            f"Val size: {n_val}\n"
            f"Test size: {n_test}")
    sys.stdout.flush() 

    #------------------#
    # Benchmark models #
    #------------------#
    bm_extractor = Extractors([[d,0] for d in dim_modalities], dim_modalities, train_loader, val_loader)
    _ = bm_extractor.get_dummy_extractors()
    bm_cohort = Cohorts(extractors=bm_extractor, combined_hidden_layers=combined_hiddens, output_dim=output_dim)

    if run_oracle:
        oracle_dims = [dim_latent[0], dim_latent[1]+dim_latent[2]]
        oracle_extractor = Extractors([[d,0] for d in oracle_dims], oracle_dims, oracle_train_loader, oracle_val_loader)
        _ = oracle_extractor.get_dummy_extractors()
        oracle_cohort = Cohorts(extractors=oracle_extractor, combined_hidden_layers=combined_hiddens, output_dim=output_dim)

    #----------------------------#
    # Proposed model: Meta Fuse  #
    #----------------------------#
    meta_cohort = Cohorts_new(dim_modalities = dim_modalities, num_modalities= num_modalities, mod_hiddens  = mod_hiddens, output_dim=output_dim)

    #------------------------------#
    #  Train and test benchmarks   #
    #------------------------------#
    bm_models = bm_cohort.get_cohort_models()
    _, bm_dims = bm_cohort.get_cohort_info()
    bm = Benchmarks(config, bm_models, bm_dims, [train_loader, val_loader])
    bm.train()
    res = bm.test(test_loader)
    res_list.append(res)
    print(f"Finished running basic benchmarks!")

    #------------------------------#
    #  Train and test Meta Fuse    #
    #------------------------------#
    cohort_models = meta_cohort.get_cohort_models()
    _, dim_pairs = meta_cohort.get_cohort_info()
    ###### Only change the two following.
    metafuse = Trainer_new(config, cohort_models, [train_loader, val_loader]) # New trainer function. 
    metafuse.train() 
    res = metafuse.test(test_loader) # No need to change test: simple_averaging in test_regresion() also # performance of each student on the test data, cohort_accuracy: automaticall printed and stored. 
    res = {f"metafusion_{k}": v for k, v in res.items()}
    res_list.append(res)
    metafuse.train_ablation() # This is just late fusion with student cohort.
    res = metafuse.test_ablation(test_loader) # I don't need this, no need to have different rhos.
    res = {f"indep_{k}": v for k, v in res.items()}
    res_list.append(res)

    best_rho['metafusion'] = metafuse.best_rho
    cohort_pairs['metafusion'] = dim_pairs
    cohort_pairs['indep'] = dim_pairs

    if "greedy_ensemble" in config["ensemble_methods"]:
        ens_idxs['metafusion_greedy_ensemble'] = metafuse.ens_idxs  

    if config['divergence_weight_type'] == "clustering":
        cluster_idxs['metafusion'] = metafuse.cluster_idxs

    print(f"Finished running meta fusion!")

    #----------------------------#
    # Proposed model: Joint train#
    #----------------------------#
    # joint_extractor = Extractors(mod_outs, dim_modalities, train_loader, val_loader)
    # if (extractor_type == 'encoder') or (extractor_type == 'separate'):
    #     _ = joint_extractor.get_encoder_extractors(mod_hiddens, separate=separate, config=extractor_config)
    # elif extractor_type == 'PCA':
    #     _ = joint_extractor.get_PCA_extractors()

    n_train = len(train_miss.dataset)
    n_val = len(val_miss.dataset)
    n_test = len(test_miss.dataset)
    n = n_train + n_val + n_test

    print(f"Finished splitting {data_name} dataset. Data information are summarized below:\n"
            f"Modality dimensions: {dim_modalities}\n"
            f"Data size: {n}\n"
            f"Train size: {n_train}\n"
            f"Val size: {n_val}\n"
            f"Test size: {n_test}")
    sys.stdout.flush() 

    joint_cohort = Cohorts_new(dim_modalities = dim_modalities, num_modalities= num_modalities, mod_hiddens  = mod_hiddens, output_dim=output_dim)

    # ------------------------------#
    #  Train and test Joint train  #
    # ------------------------------#
    cohort_models = joint_cohort.get_cohort_models()
    _, dim_pairs = joint_cohort.get_cohort_info()
    ###### Only change the two following.
    jointmodel = Trainer_Joint_new(config, cohort_models, [train_miss, val_miss]) # New trainer function. 
    jointmodel.train('marginal', missing_value=missing_value) 
    res = jointmodel.test(test_miss, missing_value=missing_value) # No need to change test: simple_averaging in test_regresion() also # performance of each student on the test data, cohort_accuracy: automaticall printed and stored. 
    res = {f"jointlearning_{k}": v for k, v in res.items()}
    res_list.append(res)
    cohort_pairs['cohort'] = dim_pairs

    if "greedy_ensemble" in config["ensemble_methods"]:
        ens_idxs['jointlearning_greedy_ensemble'] = jointmodel.ens_idxs  


    print(f"Finished running joint fusion!")

    #----------------------------#
    # Proposed model: Shapley train#
    #----------------------------#
    joint_cohort = Cohorts_new(dim_modalities = dim_modalities, num_modalities= num_modalities, mod_hiddens  = mod_hiddens, output_dim=output_dim)

    #------------------------------#
    #  Train and test shapley train  #
    #------------------------------#
    cohort_models = joint_cohort.get_cohort_models()
    _, dim_pairs = joint_cohort.get_cohort_info()
    ###### Only change the two following.
    jointmodel = Trainer_Joint_new(config, cohort_models, [train_miss, val_miss]) # New trainer function. 
    jointmodel.train('shapley', missing_value=missing_value) 
    res = jointmodel.test(test_miss, missing_value=missing_value) # No need to change test: simple_averaging in test_regresion() also # performance of each student on the test data, cohort_accuracy: automaticall printed and stored. 
    res = {f"shapley_{k}": v for k, v in res.items()}
    res_list.append(res)
    cohort_pairs['cohort'] = dim_pairs

    if "greedy_ensemble" in config["ensemble_methods"]:
        ens_idxs['shapley_greedy_ensemble'] = jointmodel.ens_idxs  


    print(f"Finished running shapley fusion!")        

    results = []
    for i, res in enumerate(res_list):
        for method, val in res.items():
            results.append({'Method': method, 'Test_metric': val, 
                            'best_rho':best_rho.get(method.split('_')[0]), 'cohort_pairs':cohort_pairs.get(method.split('_')[0]),
                            'ensemble_idxs': ens_idxs.get(method), 'cluster_idxs': cluster_idxs.get(method.split('_')[0])})

    results = pd.DataFrame(results)
    results['random_state']=random_state
    results["dim_modalities"] = [dim_modalities] * len(results)
    results['n'] = n
    results['n_train'] = n_train
    results['n_val'] = n_val
    results['n_test'] = n_test 

    return results




In [4]:
#####################
#  Run Experiments  #
#####################
results = []

for i in tqdm(range(1, repetition+1), desc="Repetitions", leave=True, position=0):
    print(f'Running with repetition {i}...')
    random_state = repetition * (seed-1) + i
    # print(random_state)
    set_random_seed(random_state)

    # Run experiment
    tmp = run_single_experiment_filter(config, extractor_config, n, random_state, mod_hiddens,
                                run_oracle=False, run_coop=True, run_all_at_once=False)
    
    results.append(tmp)


results = pd.concat(results, ignore_index=True)

add_header(results)

Repetitions:   0%|          | 0/1 [00:00<?, ?it/s]

Running with repetition 1...
Finished splitting regression dataset. Data information are summarized below:
Modality dimensions: [200, 300, 100]
Data size: 971
Train size: 619
Val size: 151
Test size: 201


/Users/parnian/Desktop/Github/Multimodal-ML/notebooks/../meta_fusion/utils.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.data = torch.tensor(data, dtype=torch.float32)


Start training benchmark models...
Training with disagreement penalty = 0

Epoch: 1/10 - LR: 0.001000


100%|██████████| 619/619 [00:00<00:00, 2304.60it/s]


...

100%|██████████| 1280/1280 [00:00<00:00, 1425.61it/s, avg_loss=249.5422, batch_time=0.045s]



Epoch: 10/10 - LR: 0.001000
Using exact Shapley computation


Repetitions: 100%|██████████| 1/1 [00:23<00:00, 23.80s/it]

Finished training student cohort!
Selecting greedy ensemble on the best cohort...
Pruned 1 worst models, keeping 2 models
Initial best models: [1, 0] with losses: [tensor(445.2227, grad_fn=<MseLossBackward0>), tensor(464.6771, grad_fn=<MseLossBackward0>)]
Done!
Method: (simple_average), Test_MSE: 288.4032287597656
Method: (weighted_average), Test_MSE: 282.1300354003906
Method: (greedy_ensemble), Test_MSE: 306.8833312988281
Method: (best_single), Test_MSE: 333.8660888671875
Method: (cohort), Test_MSE: [434.3639831542969, 332.5388488769531, 347.43487548828125]
Finished running shapley fusion!


,Method,Test_metric,best_rho,cohort_pairs,ensemble_idxs,cluster_idxs,random_state,dim_modalities,n,n_train,n_val,n_test,extractor,weight_type
0,modality_1,393.490601,NaN,None,None,None,1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
1,modality_2,369.784821,NaN,None,None,None,1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
2,modality_3,377.919128,NaN,None,None,None,1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
3,early_fusion,282.359131,NaN,None,None,None,1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
4,late_fusion,331.14917,NaN,None,None,None,1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
5,metafusion_simple_average,326.05423,0.99,"[200, 300, 100]",None,[1],1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
6,metafusion_weighted_average,324.052032,0.99,"[200, 300, 100]",None,[1],1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
7,metafusion_greedy_ensemble,318.82843,0.99,"[200, 300, 100]","[1, 2]",[1],1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
8,metafusion_best_single,358.433136,0.99,"[200, 300, 100]",None,[1],1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering
9,metafusion_cohort,"[390.05145263671875, 358.4331359863281, 365.56...",0.99,"[200, 300, 100]",None,[1],1234,"[200, 300, 100]",2000,1280,320,400,separate,clustering


In [5]:
EXPERIMENT_VARIANT = "filtered"

# Build a filename that encodes the key experiment parameters
outfile_name = (
    f"{EXPERIMENT_VARIANT}_"
    f"dim_modalities{dim_modalities}_"
    f"noise{noise_ratios}_"
    f"trans{trans_label}_"
    f"fractions{fractions}_"
    f"seed{seed}"
)
# sanitize characters that are annoying in filenames
outfile_name = (
    outfile_name.replace(" ", "")
    .replace("[", "(")
    .replace("]", ")")
    .replace(",", "-")
)

outfile = outdir + outfile_name + ".txt"
print("Output file: {:s}".format(outfile), end="\n")
sys.stdout.flush()


#####################
#    Save Results   #
#####################
results.to_csv(outfile, index=False)
print("\nResults written to {:s}\n".format(outfile))
sys.stdout.flush()

# After the job is done, remove the model directory to free up space
if os.path.exists(ckpt_dir):
    print(f"Deleting the model checkpoint directory: {ckpt_dir}")
    shutil.rmtree(ckpt_dir)
    print(f"Model checkpoint directory {ckpt_dir} has been deleted.")


Output file: ./results/regression_linear_early/filtered_dim_modalities(200-300-100)_noise(0.6-0.1-0.1)_transm1linear_m2quadratic_m3quadratic_sharedlinear_fractions(1-0.8-0.6)_seed1234.txt

Results written to ./results/regression_linear_early/filtered_dim_modalities(200-300-100)_noise(0.6-0.1-0.1)_transm1linear_m2quadratic_m3quadratic_sharedlinear_fractions(1-0.8-0.6)_seed1234.txt

Deleting the model checkpoint directory: ./checkpoints/regression_linear_early/seed1234/
Model checkpoint directory ./checkpoints/regression_linear_early/seed1234/ has been deleted.
